# 07 — Compound optimization

**Goal:** optimize the **planner** inside a `MetaOrchestrator`, and watch the whole orchestration improve.

Notebook 06 evolved one isolated agent. That is the simplest case. This notebook does something more interesting: it optimizes the **planner** — the agent that decides how to break a goal into subtasks. If the planner gets better, every spawned sub-agent receives a cleaner subtask. The whole orchestration gets sharper as a result.

The pattern is the same as notebook 06: gold set → scoring function → genome → run GEPA → commit. The difference is what we are scoring. Here we score the **shape** of the planner's output (number of subtasks, names that include the right keywords), not the final answer.

After the optimization, we will also run a held-out goal through the full orchestration **before and after** to see whether the downstream effect actually shows up.

**Before you start:** same as notebook 06. Expected cost: ~$2–$3.


## 1. Load the config and define the planner

The planner is a `BaseAgent` whose output type is `TaskDecomposition`. The initial prompt is short and neutral. We will replace it with whatever GEPA evolves.


In [1]:
from typing import Any
from pydantic import BaseModel
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState
from orqest.autonomy.meta import TaskDecomposition

config = load_config()
print(f"Model: {config.llm_model}")

INITIAL_PLANNER_PROMPT = (
    "You decompose a high-level goal into 2-4 concrete subtasks. "
    "Each subtask needs a short snake_case name, a one-sentence description, "
    "and requires_agent=True. Keep the list minimal."
)


class PlannerAgent(BaseAgent[GlobalState, TaskDecomposition]):
    async def _run_implementation(self, state: GlobalState, **kwargs: Any) -> TaskDecomposition:
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output

Model: openai:gpt-5.2


## 2. Build a planner-specific gold set

A planner's job is to produce a **good decomposition**. So the gold set is about decomposition shape, not final answers.

We use 15 examples in three buckets:

- **Single-step (5)** — the goal really only needs **one** specialist. A good planner should not over-decompose. Example: "What is 13 × 47?" should be **1–2** subtasks, not 4.
- **Multi-step (5)** — the goal genuinely needs **3+** subtasks. Example: "research, summarize, recommend."
- **Ambiguous (5)** — the goal is underspecified. A good planner should produce **2–3** subtasks where the first one **clarifies scope**, instead of fabricating details.

We don't grade exact matches on subtask names — LLMs vary on naming. Instead we check that the **count** is in the expected range and that subtask names mention at least one expected keyword.


In [2]:
from orqest.optimization import GoldExample


class PlannerExpected(BaseModel):
    min_subtasks: int
    max_subtasks: int
    must_mention: list[str] = []   # at least one of these keywords should appear in subtask names


def _ex(goal: str, lo: int, hi: int, mention: list[str], bucket: str) -> GoldExample:
    return GoldExample[str, PlannerExpected](
        input=goal,
        expected=PlannerExpected(min_subtasks=lo, max_subtasks=hi, must_mention=mention),
        id=f"{bucket}-{hash(goal) % 10000}",
    )


EXAMPLES = [
    # Single-step (1-2 subtasks).
    _ex("Translate this sentence to French: 'The cat sat on the mat.'", 1, 2, ["translat"], "single"),
    _ex("What is 13 * 47?", 1, 2, ["calc", "comput", "answer"], "single"),
    _ex("Define the word 'antidisestablishmentarianism'.", 1, 2, ["defin", "explain"], "single"),
    _ex("Sort this list alphabetically: zebra, apple, mango, banana.", 1, 2, ["sort"], "single"),
    _ex("What's the capital of France?", 1, 2, ["answer", "lookup", "capital"], "single"),
    # Multi-step (3-4 subtasks).
    _ex("Research the latest electric vehicle market, summarize the top 3 manufacturers, and recommend one for a family of 5.", 3, 4, ["research", "summar", "recommend"], "multi"),
    _ex("Build a meal plan: gather dietary requirements, propose a 7-day menu, then list the shopping items.", 3, 4, ["requir", "menu", "shop"], "multi"),
    _ex("Plan a 3-day trip to Tokyo: research attractions, build a daily itinerary, and estimate the total budget.", 3, 4, ["research", "itinerary", "budget"], "multi"),
    _ex("Diagnose a slow API: gather logs, identify the bottleneck, propose a fix, and validate against test cases.", 3, 4, ["log", "bottleneck", "fix", "valid"], "multi"),
    _ex("Design a CLI for converting CSV to JSON: name subcommands, list global flags, write a usage example.", 3, 4, ["subcommand", "flag", "example"], "multi"),
    # Ambiguous (2-3 subtasks — first one should clarify).
    _ex("Help me with my project.", 2, 3, ["clarif", "requir", "scope"], "ambiguous"),
    _ex("Make this better.", 2, 3, ["clarif", "context", "requir"], "ambiguous"),
    _ex("I need some advice.", 2, 3, ["clarif", "context", "topic"], "ambiguous"),
    _ex("Tell me about it.", 2, 3, ["clarif", "context", "subject"], "ambiguous"),
    _ex("Fix the issue.", 2, 3, ["clarif", "identif", "context"], "ambiguous"),
]
print(f"Loaded {len(EXAMPLES)} planner gold examples.")

Loaded 15 planner gold examples.


## 3. Wrap the planner in an `Evaluator`

The scoring function is two checks:

- **Count check** — is the number of subtasks within the expected range?
- **Mention check** — do the subtask names contain at least one of the expected keywords?

Both pass → 1.0. One passes → 0.5. Neither → 0.0.

This is a deliberately coarse scoring function. We're not grading the prose; we're grading the **structure** of the plan.


In [3]:
from orqest.optimization import Evaluator


def make_planner(decoded: dict[str, Any]) -> PlannerAgent:
    return PlannerAgent(
        agent_name="planner",
        system_prompt=decoded["planner.system_prompt"],
        output_type=TaskDecomposition,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )


def score(output: TaskDecomposition, ex: GoldExample[str, PlannerExpected]) -> float:
    expected = ex.expected
    n = len(output.subtasks)
    count_ok = expected.min_subtasks <= n <= expected.max_subtasks

    name_blob = " ".join(s.name.lower() for s in output.subtasks)
    mention_ok = (not expected.must_mention) or any(
        keyword in name_blob for keyword in expected.must_mention
    )

    if count_ok and mention_ok:
        return 1.0
    if count_ok or mention_ok:
        return 0.5
    return 0.0


evaluator = Evaluator(agent_factory=make_planner, score_fn=score)
print("Evaluator ready.")

Evaluator ready.


## 4. Baseline — what does the unoptimized planner do?

Run the evaluator before optimizing. This tells us where the planner already does well and where it struggles.


In [4]:
import statistics
from collections import defaultdict


async def measure(decoded: dict[str, Any]) -> dict[str, float]:
    bundles = await evaluator.evaluate_batch(decoded, EXAMPLES)
    by_bucket: dict[str, list[float]] = defaultdict(list)
    for ex, b in zip(EXAMPLES, bundles, strict=True):
        bucket = ex.id.split("-")[0]
        by_bucket[bucket].append(b.accuracy)
    return {bucket: statistics.mean(scores) for bucket, scores in by_bucket.items()} | {
        "overall": statistics.mean(b.accuracy for b in bundles)
    }


baseline = await measure({"planner.system_prompt": INITIAL_PLANNER_PROMPT})
print("Baseline (planner structural quality):")
for k, v in baseline.items():
    print(f"  {k:14s} {v:.3f}")

Baseline (planner structural quality):
  single         0.300
  multi          1.000
  ambiguous      0.900
  overall        0.733


## 5. Define the genome

One prompt slot. Note how specific the **constraints** field is — this is the optimizer's compass. We're telling it explicitly: "match depth to complexity, and for ambiguous goals the first subtask should clarify."


In [5]:
from orqest.optimization import Genome, PromptGene

genome = Genome(genes=[
    PromptGene(
        name="planner.system_prompt",
        initial=INITIAL_PLANNER_PROMPT,
        constraints=(
            "Match decomposition depth to goal complexity: 1-2 subtasks for trivial goals, "
            "3-4 for genuinely multi-step goals, 2-3 for ambiguous goals (where the first "
            "subtask should typically be clarifying scope rather than fabricating it)."
        ),
    ),
])
print("Genome ready.")

## 6. Run the optimizer

Same machinery as notebook 06. The only difference is the gold set and the scoring function — the optimizer itself doesn't know or care whether you're optimizing a planner, a summarizer, or anything else.


In [6]:
from orqest.optimization import OptimizationConfig, OptimizationRunner

opt_config = OptimizationConfig(
    max_metric_calls=80,
    reflection_model=config.llm_model,
    minibatch_size=3,
    valset_fraction=0.4,
    seed=42,
)

runner = OptimizationRunner(
    opt_config,
    genome=genome,
    evaluator=evaluator,
    api_key=config.llm_api_key,
)

result = await runner.optimize(EXAMPLES)
print(f"Best aggregate score: {result.best_score:.3f}")
print(f"Pareto frontier size: {len(result.pareto_candidates)}")

Iteration 0: Base program full valset score: 0.9145154691933324 over 6 / 6 examples
Iteration 1: Selected program 0 score: 0.9145154691933324


Iteration 1: Proposed new text for planner.system_prompt: You are a task decomposer.

Given a single high-level user goal, output a minimal JSON array of 2–4 subtasks (no extra keys, no prose, no markdown). Each subtask must be an object with:
- name: short, unique, snake_case
- description: exactly one sentence describing what to do
- requires_agent: true

Decomposition rules (match depth to complexity):
- Trivial, single-action goals (e.g., “sort this list”, “define a word”): use 1–2 subtasks total (prefer 1 when it is truly one-step).
- Ambiguous/vague goals (e.g., “Help me with my project”): use 2–3 subtasks, where the FIRST subtask is to clarify scope/requirements rather than inventing details.
- Genuinely multi-step goals: use 3–4 subtasks.

Additional constraints:
- Keep the list minimal: do not add optional/parallel tasks unless required for completion.
- Do not fabricate missing information; if needed, create a clarification subtask.
- Ensure each description is actionable and

Iteration 1: New subsample score 2.3098342190199945 is better than old score 2.2072015076000024. Continue to full eval and add to candidate pool.


Iteration 1: Valset score for new program: 0.8494511523233313 (coverage 6 / 6)
Iteration 1: Val aggregate for new program: 0.8494511523233313
Iteration 1: Individual valset scores for new program: {2: 0.9442840621599953, 3: 0.44071732028000044, 0: 0.932969179239999, 1: 0.9098137355600011, 4: 0.9344873362399995, 5: 0.934435280459993}
Iteration 1: Objective aggregate scores for new program: {'accuracy': 0.9166666666666666, 'cost_usd': 0.0, 'latency_ms': 3360.775717166765}
Iteration 1: New valset pareto front scores: {0: 0.932969179239999, 1: 0.9098137355600011, 2: 0.9442840621599953, 3: 0.9262567426999976, 4: 0.9344873362399995, 5: 0.934435280459993}
Iteration 1: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 4274.226540333378}
Iteration 1: Valset pareto front aggregate score: 0.930374389393331
Iteration 1: Updated valset pareto front programs: {0: {1}, 1: {1}, 2: {1}, 3: {0}, 4: {1}, 5: {1}}
Iteration 1: Updated objective pareto front programs: {'accurac

Iteration 2: Proposed new text for planner.system_prompt: You are a task decomposer.

Given a single high-level user goal (the “input”), output a minimal list of 2–4 concrete subtasks that would accomplish the goal.

Output format:
- Return ONLY a JSON array (no prose, no extra keys).
- Each array element is an object with exactly these keys:
  - name: short snake_case identifier
  - description: one sentence describing what to do
  - requires_agent: always true

Decomposition rules:
- Keep the list minimal and non-overlapping; each subtask should be actionable and logically ordered.
- Match decomposition depth to goal complexity:
  - Trivial goals → 1–2 subtasks (but prefer 2 when the task is underspecified, e.g., “Make this better.”)
  - Ambiguous goals → 2–3 subtasks, where the FIRST subtask is to clarify scope/constraints (do not invent specifics).
  - Genuinely multi-step goals → 3–4 subtasks.
- Do not fabricate missing context; include a clarification subtask instead.
- Avoid imp

...

Iteration 7: New subsample score 2.7440574002799987 is better than old score 1.2317128622800009. Continue to full eval and add to candidate pool.


Iteration 7: Valset score for new program: 0.9180943327099991 (coverage 6 / 6)
Iteration 7: Val aggregate for new program: 0.9180943327099991
Iteration 7: Individual valset scores for new program: {0: 0.9487941939399934, 5: 0.9078471130000071, 1: 0.895635919360002, 2: 0.8955544874000043, 3: 0.932421351819994, 4: 0.928312930739994}
Iteration 7: Objective aggregate scores for new program: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 4095.2833645000433}
Iteration 7: New valset pareto front scores: {0: 0.9652116854399992, 1: 0.9631623379600023, 2: 0.958994094059999, 3: 0.9590618006600016, 4: 0.9384639092599991, 5: 0.934435280459993}
Iteration 7: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 4274.226540333378}
Iteration 7: Valset pareto front aggregate score: 0.9532215179733323
Iteration 7: Updated valset pareto front programs: {0: {5}, 1: {5}, 2: {3}, 3: {3}, 4: {4}, 5: {1}}
Iteration 7: Updated objective pareto front programs: {'accuracy': {0, 3, 6}, 

Iteration 8: Proposed new text for planner.system_prompt: You are a task-decomposition assistant. Given a single user goal as input, output a minimal list of concrete subtasks.

REQUIREMENTS
- Produce 2–4 subtasks total, unless the goal is truly trivial:
  - Trivial one-step goals (e.g., define a word, answer a simple fact, sort a short list) MUST be decomposed into only 1–2 subtasks.
  - Ambiguous goals should use 2–3 subtasks, where the FIRST subtask is to clarify scope/requirements (do not invent details).
  - Genuinely multi-step goals should use 3–4 subtasks.
- Each subtask MUST include:
  - name: short snake_case identifier
  - description: exactly one sentence describing the action
  - requires_agent: true (always)
- Keep the list minimal: no filler, no redundant steps, no extra commentary.

OUTPUT FORMAT
Return only a JSON array of objects in the following shape (no surrounding text):
[
  {"name":"...", "description":"...", "requires_agent":true},
  ...
]

QUALITY GUIDELINES
- 

Iteration 8: New subsample score 2.5495344407800165 is better than old score 0.8096764532000088. Continue to full eval and add to candidate pool.


Iteration 8: Valset score for new program: 0.7927938189766685 (coverage 6 / 6)
Iteration 8: Val aggregate for new program: 0.7927938189766685
Iteration 8: Individual valset scores for new program: {2: 0.9406795780000083, 3: 0.9529286977600077, 4: 0.6559261650200006, 0: 0.9242822012200032, 1: 0.9078879817799952, 5: 0.37505829007999636}
Iteration 8: Objective aggregate scores for new program: {'accuracy': 0.9166666666666666, 'cost_usd': 0.0, 'latency_ms': 6193.642384499905}
Iteration 8: New valset pareto front scores: {0: 0.9652116854399992, 1: 0.9631623379600023, 2: 0.958994094059999, 3: 0.9590618006600016, 4: 0.9384639092599991, 5: 0.934435280459993}
Iteration 8: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 6193.642384499905}
Iteration 8: Valset pareto front aggregate score: 0.9532215179733323
Iteration 8: Updated valset pareto front programs: {0: {5}, 1: {5}, 2: {3}, 3: {3}, 4: {4}, 5: {1}}
Iteration 8: Updated objective pareto front programs: {'accu

## 7. Diff and per-bucket lift

See what changed in the prompt, then re-measure per bucket to find out where the optimizer helped (and where it might have hurt).


In [7]:
from orqest.optimization import apply_result

baseline_planner = make_planner({"planner.system_prompt": INITIAL_PLANNER_PROMPT})
for d in apply_result(result, target=baseline_planner):
    if d.changed:
        print(d.unified)

evolved = await measure(result.best_decoded)
print("\nPer-bucket structural quality:")
print(f"  {'bucket':<12s} {'before':>10s} {'after':>10s} {'delta':>10s}")
for bucket in ("single", "multi", "ambiguous", "overall"):
    b, a = baseline.get(bucket, 0.0), evolved.get(bucket, 0.0)
    arrow = "↑" if a > b else ("↓" if a < b else "=")
    print(f"  {bucket:<12s} {b:>10.3f} {a:>10.3f}     {a - b:+.3f} {arrow}")

--- planner.system_prompt (before)
+++ planner.system_prompt (after)
@@ -1 +1,33 @@
-You decompose a high-level goal into 2-4 concrete subtasks. Each subtask needs a short snake_case name, a one-sentence description, and requires_agent=True. Keep the list minimal.
+You are a JSON-only task decomposer.
+
+Goal:
+Given a single user goal in plain text, output a minimal decomposition as a JSON array of subtasks.
+
+Hard output constraints:
+- Output ONLY a valid JSON array (no surrounding prose, no markdown, no code fences).
+- The array must contain 1–4 objects total.
+- Each object MUST have exactly these keys (no more, no less):
+  - "name": short, unique, snake_case identifier
+  - "description": exactly one sentence describing a concrete action
+  - "requires_agent": true (boolean literal, always true)
+- Do not include empty output; always return the JSON array with the chosen subtasks.
+
+Decomposition depth rules (critical):
+- Trivial single-step requests (e.g., sorting a short l


Per-bucket structural quality:
  bucket           before      after      delta
  single            0.300      1.000     +0.700 ↑
  multi             1.000      1.000     +0.000 =
  ambiguous         0.900      0.800     -0.100 ↓
  overall           0.733      0.933     +0.200 ↑


## 8. The downstream effect — does the orchestration actually get better?

We just improved a **structural** metric (subtask count, keyword mentions). But the real question is: does the orchestration's **end-to-end behaviour** improve as a result?

Honest framing: planner quality is one of many factors. Sub-agent prompts, tool quality, model variance — they all matter too. Don't expect dramatic single-goal wins. Do expect cleaner subtask graphs and fewer retries.

We will run a held-out goal (one the optimizer **never saw**) through the full `MetaOrchestrator` twice: once with the original prompt, once with the evolved prompt.


First, build the registry, factory, and the held-out goal.

In [ ]:
from orqest.autonomy import AgentFactory, MetaOrchestrator, ToolRegistry

registry = ToolRegistry()
factory = AgentFactory(registry, default_model=config.llm_model, api_key=config.llm_api_key)

HELD_OUT_GOAL = (
    "Plan a small science fair project for an 8th-grader: pick a topic, "
    "design the experiment, list the materials, write a one-paragraph abstract."
)

print("Held-out goal defined.")

Now run the orchestration twice with the same goal — once with each planner prompt — and compare.

In [8]:
async def run_orchestration(planner_prompt: str) -> dict[str, Any]:
    planner = make_planner({"planner.system_prompt": planner_prompt})
    meta = MetaOrchestrator(planner, factory, registry, max_subtasks=5)
    res = await meta.solve(HELD_OUT_GOAL)
    return {
        "success": res.success,
        "n_subtasks": len(res.subtask_results),
        "successful_subtasks": sum(1 for r in res.subtask_results if r.success),
        "duration_ms": res.total_duration_ms,
    }


before = await run_orchestration(INITIAL_PLANNER_PROMPT)
after  = await run_orchestration(result.best_decoded["planner.system_prompt"])

print(f"BEFORE: {before}")
print(f"AFTER:  {after}")

BEFORE: {'success': True, 'n_subtasks': 4, 'successful_subtasks': 4, 'duration_ms': 19329.873791999944}
AFTER:  {'success': True, 'n_subtasks': 4, 'successful_subtasks': 4, 'duration_ms': 15052.32351199993}


## Recap

The takeaways:

1. The **optimization machinery is the same** whether you're tuning a single agent or a planner inside an orchestrator. Only the gold set and the scoring function change.
2. **Score what matters at the layer you're optimizing.** Here we scored the planner's **structure** (count, keywords), not the final answer. The end-to-end signal would have been too noisy to optimize against directly.
3. **Improvements at one layer ripple downstream.** A better planner means cleaner subtasks means sharper sub-agents. The effect is real but noisy — many other factors shape the final result.

**What's next:**

- **Compose with healing.** Feed `metacognition.confidence` into `MetricBundle.confidence`, and let the optimizer evolve a planner that is both more accurate **and** more confident in its decompositions. The plumbing already exists — pass `confidence_protocol=StructuredOutputProtocol()` to the `Evaluator`.
- **Topology search (notebook 08).** Once you have `TopologySpec`, the optimizer can mutate not only **what** the planner says, but **how the orchestrator wires** the spawned agents together (parallel vs sequential, with vs without a `RefinementLoop`). Same Pareto contract; a much larger search space.
- **`docs/concepts/optimization.md`** for the full architecture.
